# Case Study 01: Insurance Pricing with Gamma GLM
## Aurora-GLM Showcase: Gamma GLM for Right-Skewed Positive Continuous Data

---

## Overview

This case study demonstrates **Gamma GLM** for modeling medical insurance charges — positive, right-skewed cost data where the Gamma distribution is the natural choice over Gaussian regression.

### Research Context

Medical costs vary substantially across individuals due to:
- **Demographics**: Age, sex, BMI
- **Lifestyle**: Smoking status
- **Family**: Number of dependents
- **Geography**: Regional cost differences

### Research Questions

1. **RQ1:** Which factors most strongly predict medical costs?
2. **RQ2:** Why is Gamma GLM superior to Gaussian GLM for cost data?
3. **RQ3:** What is the multiplicative effect of smoking on costs?
4. **RQ4:** Which link function fits better — log (multiplicative) or identity (additive)?
5. **RQ5:** Is there an age × BMI interaction effect on charges?
6. **RQ6:** How do effects combine for specific customer profiles?

### Aurora-GLM Capabilities Demonstrated

1. Gamma GLM with log and identity links
2. Comparison with Gaussian GLM
3. Interaction terms (age × BMI)
4. Multiplicative effect interpretation (rate ratios)
5. Residual diagnostics for distribution choice
6. Model selection (AIC/BIC)
7. Multi-backend comparison

---

**Dataset**: 1,338 medical insurance records (age, sex, BMI, children, smoker, region, charges).

## PART I: Setup and Data Loading

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import time
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

# Aurora-GLM
from aurora.models.glm import fit_glm

# Check for PyTorch
try:
    import torch
    TORCH_AVAILABLE = True
    GPU_AVAILABLE = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'N/A'
except ImportError:
    TORCH_AVAILABLE = False
    GPU_AVAILABLE = False
    GPU_NAME = 'N/A'

# Configure visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
sns.set_context('notebook', font_scale=1.1)
%config InlineBackend.figure_format = 'retina'

np.random.seed(42)

print("="*80)
print("ENVIRONMENT SETUP")
print("="*80)
print(f"\nBackend Availability:")
print(f"   NumPy: Available")
print(f"   PyTorch: {'Available' if TORCH_AVAILABLE else 'Not installed'}")
print(f"   GPU: {'Available - ' + GPU_NAME if GPU_AVAILABLE else 'Not available'}")
print("\n" + "="*80)

In [ ]:
# Load Medical Insurance data
data_path = Path('data/insurance.csv')

if not data_path.exists():
    print("Downloading Medical Insurance data...")
    import urllib.request
    data_path.parent.mkdir(exist_ok=True)
    url = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
    urllib.request.urlretrieve(url, data_path)
    print(f"Downloaded to {data_path}")

df = pd.read_csv(data_path)

print("="*80)
print("DATA LOADING")
print("="*80)
print(f"\nDataset:")
print(f"   Observations: {len(df):,}")
print(f"   Variables: {len(df.columns)}")
print(f"\nVariables: {list(df.columns)}")
print(f"\nTarget: charges (annual medical costs in USD)")
print("\n" + "="*80)

In [ ]:
# Data preprocessing
print("="*80)
print("PREPROCESSING")
print("="*80)

# Summary of variables
print("\nVariable Summary:")
print(f"   age: {df['age'].min()}-{df['age'].max()} years (mean={df['age'].mean():.1f})")
print(f"   sex: {df['sex'].value_counts().to_dict()}")
print(f"   bmi: {df['bmi'].min():.1f}-{df['bmi'].max():.1f} (mean={df['bmi'].mean():.1f})")
print(f"   children: {df['children'].min()}-{df['children'].max()} (mean={df['children'].mean():.1f})")
print(f"   smoker: {df['smoker'].value_counts().to_dict()}")
print(f"   region: {df['region'].nunique()} categories")

# Target distribution
print(f"\nCharges Distribution:")
print(f"   Mean: ${df['charges'].mean():,.2f}")
print(f"   Median: ${df['charges'].median():,.2f}")
print(f"   Std: ${df['charges'].std():,.2f}")
print(f"   Min: ${df['charges'].min():,.2f}")
print(f"   Max: ${df['charges'].max():,.2f}")
print(f"   Skewness: {df['charges'].skew():.2f} (highly right-skewed)")

# Gamma distribution requirements
print(f"\nGamma distribution requirements:")
print(f"   All values positive: {(df['charges'] > 0).all()}")
print(f"   Right-skewed: {df['charges'].skew() > 0}")
print(f"   Variance increases with mean: checked in residual diagnostics (Part V)")

# Create dummy variables
df['male'] = (df['sex'] == 'male').astype(int)
df['smoker_yes'] = (df['smoker'] == 'yes').astype(int)

# Region dummies (reference: northeast)
region_dummies = pd.get_dummies(df['region'], prefix='region', drop_first=True)
df = pd.concat([df, region_dummies], axis=1)

# Standardize continuous variables (coefficients per SD, comparable magnitudes)
df['age_std'] = (df['age'] - df['age'].mean()) / df['age'].std()
df['bmi_std'] = (df['bmi'] - df['bmi'].mean()) / df['bmi'].std()

print(f"\nCreated dummy variables for sex, smoker, region")
print(f"Standardized age and bmi")
print("\n" + "="*80)

## PART II: Exploratory Data Analysis

In [ ]:
print("="*80)
print("EXPLORATORY DATA ANALYSIS")
print("="*80)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Panel 1: Distribution of charges
axes[0, 0].hist(df['charges'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['charges'].mean(), color='red', linestyle='--', linewidth=2,
                   label=f"Mean=${df['charges'].mean()/1000:.1f}K")
axes[0, 0].axvline(df['charges'].median(), color='orange', linestyle='--', linewidth=2,
                   label=f"Median=${df['charges'].median()/1000:.1f}K")
axes[0, 0].set_xlabel('Annual Medical Charges ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Medical Charges', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Panel 2: Log-transformed charges
axes[0, 1].hist(np.log(df['charges']), bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Log(Charges)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Log-Transformed Charges', fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# Panel 3: Charges by smoker status
df.boxplot(column='charges', by='smoker', ax=axes[0, 2])
axes[0, 2].set_xlabel('Smoker')
axes[0, 2].set_ylabel('Charges ($)')
axes[0, 2].set_title('Charges by Smoking Status', fontweight='bold')
plt.suptitle('')

# Panel 4: Charges vs Age
colors = ['blue' if s == 'no' else 'red' for s in df['smoker']]
axes[1, 0].scatter(df['age'], df['charges'], c=colors, alpha=0.5, s=20)
axes[1, 0].set_xlabel('Age')
axes[1, 0].set_ylabel('Charges ($)')
axes[1, 0].set_title('Charges vs Age (Blue=Non-smoker, Red=Smoker)', fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Panel 5: Charges vs BMI
axes[1, 1].scatter(df['bmi'], df['charges'], c=colors, alpha=0.5, s=20)
axes[1, 1].set_xlabel('BMI')
axes[1, 1].set_ylabel('Charges ($)')
axes[1, 1].set_title('Charges vs BMI (Blue=Non-smoker, Red=Smoker)', fontweight='bold')
axes[1, 1].grid(alpha=0.3)

# Panel 6: Charges by region
df.boxplot(column='charges', by='region', ax=axes[1, 2])
axes[1, 2].set_xlabel('Region')
axes[1, 2].set_ylabel('Charges ($)')
axes[1, 2].set_title('Charges by Region', fontweight='bold')
plt.suptitle('')

plt.tight_layout()
plt.show()

# Summary by smoker
print("\nCharges by Smoking Status:")
smoker_summary = df.groupby('smoker')['charges'].agg(['mean', 'median', 'std'])
print(smoker_summary.round(2))
ratio = df[df['smoker']=='yes']['charges'].mean() / df[df['smoker']=='no']['charges'].mean()
print(f"\nSmoker/Non-smoker ratio: {ratio:.2f}x")

print("\n" + "="*80)

## PART III: Mathematical Specification

### Why Gamma GLM for Cost Data?

Medical costs have three key properties:
1. **Strictly positive**: Costs cannot be negative
2. **Right-skewed**: Few very high costs, many moderate costs
3. **Variance increases with mean**: Higher costs have higher variability

### Gamma Distribution

$$Y \sim \text{Gamma}(\alpha, \beta)$$

With:
- $E(Y) = \mu = \alpha/\beta$
- $\text{Var}(Y) = \mu^2/\alpha = \phi \mu^2$

The variance is proportional to $\mu^2$, making it suitable for cost data.

### Gamma GLM with Log Link

**Model:**
$$Y_i \sim \text{Gamma}(\mu_i, \phi)$$

**Link Function:**
$$\log(\mu_i) = \mathbf{x}_i^T \boldsymbol{\beta}$$

**Inverse Link:**
$$\mu_i = \exp(\mathbf{x}_i^T \boldsymbol{\beta})$$

### Multiplicative Interpretation

For predictor $x_j$:
$$\frac{E(Y|x_j + 1)}{E(Y|x_j)} = e^{\beta_j}$$

A one-unit increase in $x_j$ **multiplies** the expected cost by $e^{\beta_j}$ (the **rate ratio**).

### Link Function Choice: Log vs Identity

| Property | Log link | Identity link |
|----------|----------|---------------|
| Predictor | $\log(\mu) = \mathbf{x}^T\boldsymbol{\beta}$ | $\mu = \mathbf{x}^T\boldsymbol{\beta}$ |
| Effects | Multiplicative ($e^{\beta_j}$ = rate ratio) | Additive ($\beta_j$ in dollars) |
| Positivity | Guaranteed ($\mu = e^\eta > 0$) | Not guaranteed during fitting |
| Best when | Effects scale with the mean | Effects are roughly constant in $ |

We fit **both** and compare by AIC (Part IV); we interpret the log-link model (rate ratios) regardless, since multiplicative effects are the natural language of insurance pricing.

### Interaction Term (Age × BMI)

The main-effects model assumes age and BMI act independently. To test whether the BMI effect changes with age we add a product term (on the standardized scales):

$$\log(\mu_i) = \beta_0 + \beta_{\text{age}} \text{age}_i^* + \beta_{\text{bmi}} \text{bmi}_i^* + \beta_{\text{age}\times\text{bmi}} \, \text{age}_i^* \text{bmi}_i^* + \ldots$$

$\beta_{\text{age}\times\text{bmi}} > 0$ means the BMI penalty **grows with age** (synergistic effect).

### Comparison with Gaussian GLM

| Property | Gaussian | Gamma (log link) |
|----------|----------|------------------|
| Variance | Constant | Proportional to $\mu^2$ |
| Support | $(-\infty, \infty)$ | $(0, \infty)$ |
| Effect interpretation | Additive | Multiplicative |
| Skewness | None | Right-skewed |

---

## PART IV: Model Fitting

In [ ]:
# Prepare design matrix
# (no explicit intercept column: fit_glm adds one via fit_intercept=True)
print("="*80)
print("PREPARING DESIGN MATRIX")
print("="*80)

X = np.column_stack([
    df['age_std'].values,       # Age (standardized)
    df['male'].values,          # Sex
    df['bmi_std'].values,       # BMI (standardized)
    df['children'].values,      # Children
    df['smoker_yes'].values,    # Smoker
    df['region_northwest'].values,
    df['region_southeast'].values,
    df['region_southwest'].values
])

y = df['charges'].values

predictor_names = ['Age (std)', 'Male', 'BMI (std)', 'Children', 'Smoker',
                   'Northwest', 'Southeast', 'Southwest']

print(f"\nDesign Matrix: {X.shape}")
print(f"Predictors: {predictor_names}")
print("\n" + "="*80)

In [ ]:
print("="*80)
print("MODEL 1: GAUSSIAN GLM (Baseline)")
print("="*80)

start_time = time.time()
result_gaussian = fit_glm(
    X=X,
    y=y,
    family='gaussian',
    link='identity'
)
time_gaussian = time.time() - start_time

print(f"\nConverged: {result_gaussian.converged_}")
print(f"Fitting time: {time_gaussian:.3f} seconds")
print(f"\nModel Fit:")
print(f"   Deviance: {result_gaussian.deviance_:.2f}")
print(f"   AIC: {result_gaussian.aic_:.2f}")
print(f"   BIC: {result_gaussian.bic_:.2f}")

print(f"\nCoefficients (Additive Effects):")
print(f"   {'Intercept':15s}: ${result_gaussian.intercept_:,.2f}")
for name, coef in zip(predictor_names, result_gaussian.coef_):
    print(f"   {name:15s}: ${coef:+,.2f}")

print("\n" + "="*80)

In [ ]:
print("="*80)
print("MODEL 2: GAMMA GLM (Log Link)")
print("="*80)

start_time = time.time()
result_gamma = fit_glm(
    X=X,
    y=y,
    family='gamma',
    link='log'
)
time_gamma = time.time() - start_time

print(f"\nConverged: {result_gamma.converged_}")
print(f"Iterations: {result_gamma.n_iter_}")
print(f"Fitting time: {time_gamma:.3f} seconds")
print(f"\nModel Fit:")
print(f"   Deviance: {result_gamma.deviance_:.2f}")
print(f"   AIC: {result_gamma.aic_:.2f}")
print(f"   BIC: {result_gamma.bic_:.2f}")

print(f"\nCoefficients (Multiplicative Effects):")
print(f"   {'Intercept':15s}: coef={result_gamma.intercept_:+.4f}, exp(coef)=${np.exp(result_gamma.intercept_):,.2f}")
for name, coef in zip(predictor_names, result_gamma.coef_):
    mult = np.exp(coef)
    pct_change = (mult - 1) * 100
    print(f"   {name:15s}: coef={coef:+.4f}, mult={mult:.3f} ({pct_change:+.1f}%)")

print("\n" + "="*80)

In [ ]:
print("="*80)
print("MODEL 3: GAMMA GLM (Identity Link) - link comparison")
print("="*80)

# Identity link: additive effects in dollars on the same Gamma family
result_identity = fit_glm(
    X=X,
    y=y,
    family='gamma',
    link='identity'
)

print(f"\nConverged: {result_identity.converged_}")
print(f"   Deviance: {result_identity.deviance_:.2f}")
print(f"   AIC: {result_identity.aic_:.2f}")
print(f"   BIC: {result_identity.bic_:.2f}")

print(f"\nLink Function Comparison (same Gamma family, same predictors):")
print(f"   AIC (log link):      {result_gamma.aic_:.2f}")
print(f"   AIC (identity link): {result_identity.aic_:.2f}")
winner = "Log" if result_gamma.aic_ < result_identity.aic_ else "Identity"
print(f"\n   {winner} link is preferred (lower AIC)")

print(f"\nCoefficients (Additive Effects, dollars):")
print(f"   {'Intercept':15s}: ${result_identity.intercept_:,.2f}")
for name, coef in zip(predictor_names, result_identity.coef_):
    print(f"   {name:15s}: ${coef:+,.2f}")

print("\nNote: interpretation below uses the log-link model (rate ratios are the")
print("natural language of insurance pricing); the identity link is reported for")
print("the link-choice question (RQ4).")

print("\n" + "="*80)

In [ ]:
print("="*80)
print("MODEL 4: GAMMA GLM (Log Link) + AGE x BMI INTERACTION")
print("="*80)

# Add the interaction of the standardized continuous predictors
X_int = np.column_stack([X, df['age_std'].values * df['bmi_std'].values])
predictor_names_int = predictor_names + ['Age x BMI']

result_int = fit_glm(
    X=X_int,
    y=y,
    family='gamma',
    link='log'
)

print(f"\nConverged: {result_int.converged_}")
print(f"   Deviance: {result_int.deviance_:.2f}")
print(f"   AIC: {result_int.aic_:.2f}")
print(f"   BIC: {result_int.bic_:.2f}")

# Significance of the interaction term
b_int = result_int.coef_[-1]
se_int = result_int.std_errors_[-1]
z_int = b_int / se_int
p_int = 2 * (1 - stats.norm.cdf(abs(z_int)))

print(f"\nInteraction term (Age x BMI):")
print(f"   coef = {b_int:+.4f}  (SE = {se_int:.4f}, z = {z_int:+.2f}, p = {p_int:.4f})")
print(f"   rate ratio = {np.exp(b_int):.3f}")
if p_int < 0.05:
    direction = "grows" if b_int > 0 else "shrinks"
    print(f"   -> SIGNIFICANT: the BMI effect {direction} with age (synergistic)")
else:
    print(f"   -> NOT significant: age and BMI act ~independently")

print(f"\n   AIC without interaction: {result_gamma.aic_:.2f}")
print(f"   AIC with interaction:    {result_int.aic_:.2f}")
print(f"   Delta AIC:               {result_int.aic_ - result_gamma.aic_:+.2f}")

print("\n" + "="*80)

In [ ]:
print("="*80)
print("MODEL COMPARISON")
print("="*80)

models = {
    'Gaussian (identity)': result_gaussian,
    'Gamma (log)': result_gamma,
    'Gamma (identity)': result_identity,
    'Gamma (log) + age:bmi': result_int,
}

print(f"\n{'Model':<26} {'Deviance':>18} {'AIC':>12} {'BIC':>12}")
print("-" * 70)
for name, res in models.items():
    print(f"{name:<26} {res.deviance_:>18.2f} {res.aic_:>12.2f} {res.bic_:>12.2f}")

print("\nNotes:")
print("   - Within the Gamma family, AIC ranks the link and the interaction")
print("     (same likelihood, same data).")
print("   - AIC comparison ACROSS families (Gaussian vs Gamma) requires caution:")
print("     the likelihoods live on different scales; the residual diagnostics")
print("     in Part V are the more reliable guide.")

print("\n" + "="*80)

## PART V: Residual Diagnostics

In [ ]:
print("="*80)
print("RESIDUAL DIAGNOSTICS: GAUSSIAN vs GAMMA")
print("="*80)

# Fitted values
mu_gaussian = result_gaussian.predict(X)
mu_gamma = result_gamma.predict(X)

# Residuals
resid_gaussian = y - mu_gaussian
resid_gamma = y - mu_gamma

# Standardized residuals
std_resid_gaussian = resid_gaussian / resid_gaussian.std()
std_resid_gamma = resid_gamma / resid_gamma.std()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Row 1: Gaussian GLM
axes[0, 0].scatter(mu_gaussian, resid_gaussian, alpha=0.5, s=20)
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Gaussian: Residuals vs Fitted', fontweight='bold')
axes[0, 0].grid(alpha=0.3)

stats.probplot(std_resid_gaussian, dist='norm', plot=axes[0, 1])
axes[0, 1].set_title('Gaussian: Q-Q Plot', fontweight='bold')

axes[0, 2].hist(std_resid_gaussian, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 2].set_xlabel('Standardized Residuals')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].set_title('Gaussian: Residual Distribution', fontweight='bold')
axes[0, 2].grid(alpha=0.3)

# Row 2: Gamma GLM
axes[1, 0].scatter(mu_gamma, resid_gamma, alpha=0.5, s=20)
axes[1, 0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Fitted Values')
axes[1, 0].set_ylabel('Residuals')
axes[1, 0].set_title('Gamma: Residuals vs Fitted', fontweight='bold')
axes[1, 0].grid(alpha=0.3)

stats.probplot(std_resid_gamma, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Gamma: Q-Q Plot', fontweight='bold')

axes[1, 2].hist(std_resid_gamma, bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1, 2].set_xlabel('Standardized Residuals')
axes[1, 2].set_ylabel('Frequency')
axes[1, 2].set_title('Gamma: Residual Distribution', fontweight='bold')
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nReading the diagnostics:")
print("   Gaussian - residual spread grows with fitted value (heteroscedasticity),")
print("              strong right skew in the residual distribution and Q-Q tails.")
print("   Gamma    - variance pattern matches Var proportional to mu^2 better;")
print("              residuals remain right-skewed (cost data) but less skewed.")

print("\n" + "="*80)

In [ ]:
print("="*80)
print("PERFORMANCE AND DIAGNOSTICS: GAMMA GLM (log link)")
print("="*80)

pred = result_gamma.predict(X)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Actual vs Predicted
ax1 = axes[0, 0]
ax1.scatter(y, pred, alpha=0.4, s=30, edgecolors='black', linewidth=0.5)
ax1.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2.5, label='Perfect prediction')
ax1.set_xlabel('Actual Charges ($)')
ax1.set_ylabel('Predicted Charges ($)')
ax1.set_title('Actual vs Predicted Charges', fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ss_res = np.sum((y - pred) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r2 = 1 - (ss_res / ss_tot)
ax1.text(0.05, 0.95, f'R² = {r2:.3f}', transform=ax1.transAxes,
         fontsize=12, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

# Plot 2: Residuals vs Fitted with smoothed trend
ax2 = axes[0, 1]
residuals = y - pred
ax2.scatter(pred, residuals, alpha=0.4, s=30, edgecolors='black', linewidth=0.5)
ax2.axhline(y=0, color='r', linestyle='--', lw=2)
from scipy.ndimage import uniform_filter1d
sorted_idx = np.argsort(pred)
smoothed = uniform_filter1d(residuals[sorted_idx], size=50)
ax2.plot(pred[sorted_idx], smoothed, 'b-', lw=2, label='Smoothed trend')
ax2.set_xlabel('Fitted Values ($)')
ax2.set_ylabel('Residuals ($)')
ax2.set_title('Residuals vs Fitted Values', fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Plot 3: Q-Q plot
ax3 = axes[0, 2]
stats.probplot(residuals, dist="norm", plot=ax3)
ax3.set_title('Q-Q Plot (Residuals)', fontweight='bold')
ax3.grid(True, alpha=0.3)

# Plot 4: Distribution of charges by smoker status
ax4 = axes[1, 0]
smokers = df[df['smoker'] == 'yes']['charges']
non_smokers = df[df['smoker'] == 'no']['charges']
ax4.hist(non_smokers, bins=30, alpha=0.7, label=f'Non-smokers (n={len(non_smokers)})',
         color='blue', edgecolor='black')
ax4.hist(smokers, bins=30, alpha=0.7, label=f'Smokers (n={len(smokers)})',
         color='red', edgecolor='black')
ax4.axvline(non_smokers.mean(), color='blue', linestyle='--', lw=2,
            label=f'Mean non-smoker: ${non_smokers.mean():,.0f}')
ax4.axvline(smokers.mean(), color='red', linestyle='--', lw=2,
            label=f'Mean smoker: ${smokers.mean():,.0f}')
ax4.set_xlabel('Charges ($)')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of Charges by Smoking Status', fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3, axis='y')

# Plot 5: Predicted charges by age and smoker status
ax5 = axes[1, 1]
age_grid = np.linspace(df['age'].min(), df['age'].max(), 100)
age_grid_std = (age_grid - df['age'].mean()) / df['age'].std()

# Reference profile: median BMI, no children, male, northeast
X_profile = np.column_stack([
    age_grid_std,
    np.ones(100),                                   # male
    np.zeros(100),                                  # median BMI (std = 0)
    np.zeros(100),                                  # no children
    np.zeros(100),                                  # non-smoker
    np.zeros(100), np.zeros(100), np.zeros(100)     # northeast
])
pred_nonsmoker = result_gamma.predict(X_profile)
X_profile_sm = X_profile.copy()
X_profile_sm[:, 4] = 1                              # smoker
pred_smoker = result_gamma.predict(X_profile_sm)

ax5.plot(age_grid, pred_nonsmoker, 'b-', linewidth=3, label='Non-smoker', alpha=0.8)
ax5.plot(age_grid, pred_smoker, 'r-', linewidth=3, label='Smoker', alpha=0.8)
ax5.fill_between(age_grid, pred_nonsmoker, pred_smoker, alpha=0.2, color='gray',
                 label='Smoking effect')
ax5.set_xlabel('Age')
ax5.set_ylabel('Predicted Charges ($)')
ax5.set_title('Predicted Charges by Age and Smoking Status', fontweight='bold')
ax5.legend(fontsize=10)
ax5.grid(True, alpha=0.3)

# Plot 6: Feature importance (standardized coefficients)
ax6 = axes[1, 2]
# Standardize all predictors (including dummies) for a fair comparison
X_std_full = (X - X.mean(axis=0)) / X.std(axis=0)
result_std = fit_glm(X_std_full, y, family='gamma', link='log')
importance = np.abs(result_std.coef_)
colors_imp = ['red' if c > 0 else 'blue' for c in result_std.coef_]

ax6.barh(predictor_names, importance, color=colors_imp, edgecolor='black', alpha=0.7)
ax6.set_xlabel('|Standardized Coefficient|')
ax6.set_title('Feature Importance (Standardized)', fontweight='bold')
ax6.grid(True, alpha=0.3, axis='x')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='red', label='Increases charges'),
                   Patch(facecolor='blue', label='Decreases charges')]
ax6.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.show()

# Performance metrics
rmse = np.sqrt(np.mean((y - pred) ** 2))
mae = np.mean(np.abs(y - pred))
mape = np.mean(np.abs((y - pred) / y)) * 100

smoker_mask = df['smoker_yes'] == 1
rmse_smoker = np.sqrt(np.mean((y[smoker_mask] - pred[smoker_mask]) ** 2))
rmse_nonsmoker = np.sqrt(np.mean((y[~smoker_mask] - pred[~smoker_mask]) ** 2))

print(f'\nOverall:')
print(f'   RMSE: ${rmse:,.2f}')
print(f'   MAE:  ${mae:,.2f}')
print(f'   MAPE: {mape:.2f}%')
print(f'   R2:   {r2:.4f}')
print(f'   Deviance: {result_gamma.deviance_:.2f}')

print(f'\nBy Smoking Status:')
print(f'   RMSE (smokers):     ${rmse_smoker:,.2f}')
print(f'   RMSE (non-smokers): ${rmse_nonsmoker:,.2f}')
print('   (smoker charges are more heterogeneous - harder to predict)')

print("\n" + "="*80)

## PART VI: Effect Interpretation

In [ ]:
print("="*80)
print("EFFECT INTERPRETATION (Gamma GLM, log link)")
print("="*80)

print("\nMultiplicative Effects on Medical Costs (rate ratios with 95% CI):")
print("-" * 70)

for i, (name, coef, se) in enumerate(zip(predictor_names, result_gamma.coef_,
                                          result_gamma.std_errors_)):
    mult = np.exp(coef)
    ci_lower = np.exp(coef - 1.96 * se)
    ci_upper = np.exp(coef + 1.96 * se)
    z_val = coef / se
    p_val = 2 * (1 - stats.norm.cdf(abs(z_val)))

    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    pct = (mult - 1) * 100

    print(f"{name:15s}: RR={mult:.3f} (95% CI: {ci_lower:.3f}-{ci_upper:.3f}) {sig}")
    print(f"{'':15s}  -> {pct:+.1f}% change in expected cost")

print("\n" + "-" * 70)
print("Significance: * p<0.05, ** p<0.01, *** p<0.001")
print("Age and BMI are standardized: effects are per standard deviation")
print(f"(1 SD age = {df['age'].std():.1f} years, 1 SD BMI = {df['bmi'].std():.1f} units)")

print("\n" + "="*80)

In [ ]:
print("="*80)
print("PRACTICAL INTERPRETATION")
print("="*80)

smoker_mult = np.exp(result_gamma.coef_[4])   # Smoker coefficient
age_mult = np.exp(result_gamma.coef_[0])      # Age (per SD)
bmi_mult = np.exp(result_gamma.coef_[2])      # BMI (per SD)

print("\n1. SMOKING EFFECT")
print(f"   Multiplier: {smoker_mult:.2f}x")
print(f"   Smokers pay {(smoker_mult-1)*100:.0f}% more than non-smokers")
print(f"   For $10,000 base cost: +${(smoker_mult-1)*10000:,.0f}")

print("\n2. AGE EFFECT (per standard deviation)")
print(f"   Multiplier: {age_mult:.3f} (per {df['age'].std():.1f} years)")
print(f"   Each SD increase in age raises costs by {(age_mult-1)*100:.1f}%")

print("\n3. BMI EFFECT (per standard deviation)")
print(f"   Multiplier: {bmi_mult:.3f} (per {df['bmi'].std():.1f} BMI units)")
print(f"   Each SD increase in BMI raises costs by {(bmi_mult-1)*100:.1f}%")

print("\n4. COMBINED EFFECTS (multiplicative scale)")
combined_mult = (age_mult ** 1.5) * smoker_mult * (bmi_mult ** 1)
print(f"   50-yr smoker, high BMI vs 30-yr non-smoker:")
print(f"   Combined multiplier: {combined_mult:.2f}x")

base_cost = np.exp(result_gamma.intercept_)
print(f"\n   Baseline (reference individual): ${base_cost:,.0f}")
print(f"   Smoker: ${base_cost * smoker_mult:,.0f}")
print(f"   Older + Smoker + Higher BMI: ${base_cost * combined_mult:,.0f}")

print("\n5. PREDICTED CHARGES FOR SPECIFIC PROFILES")
print("-" * 70)
age_mean, age_sd = df['age'].mean(), df['age'].std()
bmi_mean, bmi_sd = df['bmi'].mean(), df['bmi'].std()

profiles = [
    {'name': 'Young non-smoker',       'age': 25, 'bmi': 25, 'children': 0, 'male': 1, 'smoker': 0},
    {'name': 'Young smoker',           'age': 25, 'bmi': 25, 'children': 0, 'male': 1, 'smoker': 1},
    {'name': 'Middle-aged non-smoker', 'age': 45, 'bmi': 30, 'children': 2, 'male': 0, 'smoker': 0},
    {'name': 'Middle-aged smoker',     'age': 45, 'bmi': 30, 'children': 2, 'male': 0, 'smoker': 1},
    {'name': 'Senior non-smoker',      'age': 60, 'bmi': 28, 'children': 0, 'male': 1, 'smoker': 0},
    {'name': 'Senior smoker',          'age': 60, 'bmi': 35, 'children': 0, 'male': 1, 'smoker': 1},
]

print(f"\n{'Profile':<25} {'Predicted Charges':<20} {'vs same-age non-smoker'}")
print("-" * 70)

for profile in profiles:
    X_p = np.array([[
        (profile['age'] - age_mean) / age_sd,
        profile['male'],
        (profile['bmi'] - bmi_mean) / bmi_sd,
        profile['children'],
        profile['smoker'],
        0, 0, 0  # Northeast (reference)
    ]])
    pred_p = result_gamma.predict(X_p)[0]
    comment = "baseline" if profile['smoker'] == 0 else f"{smoker_mult:.1f}x higher"
    print(f"{profile['name']:<25} ${pred_p:>15,.2f}     {comment}")

print("\nNote the dramatic impact of smoking status across all age groups")

print("\n" + "="*80)

## PART VII: Multi-Backend Performance

In [ ]:
print("="*80)
print("MULTI-BACKEND PERFORMANCE")
print("="*80)

benchmark_results = []

# NumPy
benchmark_results.append({
    'Backend': 'NumPy (CPU)',
    'Time (s)': f'{time_gamma:.3f}',
    'Converged': 'Yes' if result_gamma.converged_ else 'No'
})

# PyTorch CPU
if TORCH_AVAILABLE:
    start_time = time.time()
    result_torch = fit_glm(X=X, y=y, family='gamma', link='log',
                          backend='torch', device='cpu')
    time_torch = time.time() - start_time
    benchmark_results.append({
        'Backend': 'PyTorch (CPU)',
        'Time (s)': f'{time_torch:.3f}',
        'Converged': 'Yes' if result_torch.converged_ else 'No'
    })

# PyTorch GPU
if GPU_AVAILABLE:
    start_time = time.time()
    result_gpu = fit_glm(X=X, y=y, family='gamma', link='log',
                        backend='torch', device='cuda')
    time_gpu = time.time() - start_time
    benchmark_results.append({
        'Backend': 'PyTorch (GPU)',
        'Time (s)': f'{time_gpu:.3f}',
        'Converged': 'Yes' if result_gpu.converged_ else 'No'
    })
    print(f"\nNote: Small dataset ({len(y):,} rows) - GPU overhead may exceed benefit")

print("\nBenchmark Results:")
benchmark_df = pd.DataFrame(benchmark_results)
print(benchmark_df.to_string(index=False))

print("\n" + "="*80)

## PART VIII: Conclusions

### Main Findings

1. **Smoking is the dominant risk factor**: rate ratio 4.48 (95% CI: 4.19-4.80) — smokers pay **+348%** more than comparable non-smokers (e.g., $12,620 → $62,455 for a 60-year-old profile).
2. **Age and BMI have significant positive effects**: +49.5% per SD of age (14 years) and +9.0% per SD of BMI (6.1 units).
3. **Children** adds +8.8% per child; **Southeast/Southwest** regions are ~13% cheaper than the Northeast reference; **sex** is marginal (-5.5%, p < 0.05 only at the 5% level).
4. **Link function (RQ4)**: the **identity link** fits better (AIC 26,246.73 vs 26,377.65 for log, Δ = 131) — additive dollar effects describe this dataset slightly better than multiplicative ones. We still interpret the log-link model because rate ratios are the natural language of pricing.
5. **Age × BMI interaction (RQ5)**: significant (coef = −0.036, p = 0.009) but **negative** — the BMI penalty *shrinks* with age — and small (ΔAIC = −4.6). The main-effects model is a reasonable working approximation.
6. **Model fit**: R² = 0.588, RMSE = $7,771, MAPE = 38.3%. Prediction error is much larger for smokers (RMSE $14,559) than non-smokers ($4,620) — smoker charges are intrinsically more heterogeneous.

### Why Gamma GLM is Better for Cost Data

1. **Respects positivity**: cannot predict negative costs
2. **Handles skewness**: natural for right-skewed distributions
3. **Variance function**: Var(Y) ∝ μ² matches real cost behavior (the Gaussian residual plots show the classic fan shape)
4. **Multiplicative interpretation**: rate ratios map directly to pricing surcharges

### When to Use Gamma GLM

- ✅ Positive continuous outcomes (costs, claims, durations, waiting times)
- ✅ Right-skewed distributions with variance increasing with the mean
- ✅ Multiplicative or additive effects (choose the link by AIC)
- ❌ Data with exact zeros (use Tweedie), binary outcomes (Binomial), or counts (Poisson/NB)

### Aurora-GLM Capabilities Demonstrated

1. Gamma GLM with log and identity links
2. Gaussian GLM baseline for comparison
3. Interaction terms (age × BMI) with Wald tests from `std_errors_`
4. Effect interpretation on the multiplicative scale (rate ratios + CIs)
5. Residual diagnostics and performance metrics by subgroup
6. Multi-backend support (NumPy here; PyTorch where installed)

### Limitations and Future Work

- **R² = 0.59**: a substantial share of cost variance is unexplained (missing medical history, occupation, lifestyle)
- **Single interaction tested**: smoker × BMI and smoker × age are plausible next candidates (the smoker subgroup's heterogeneity suggests interaction or separate models)
- **Linear continuous effects**: GAM smooths s(age), s(bmi) could capture thresholds (e.g., BMI > 30 obesity)
- **Cross-sectional**: no temporal dimension
- **Cross-validation**: out-of-sample predictive accuracy not assessed here

---

### References

- McCullagh, P., & Nelder, J. A. (1989). *Generalized Linear Models* (2nd ed.). Chapman and Hall.
- Faraway, J. J. (2016). *Extending the Linear Model with R*. CRC Press.
- Dataset source: Machine Learning with R datasets (GitHub)

---

**Analysis completed using Aurora-GLM**

**Dataset**: Medical Insurance Charges (N = 1,338)

**Models**: Gaussian GLM, Gamma GLM (log and identity links), Gamma GLM + age×BMI interaction

**Key Results**: Smokers pay 4.48× more (p < 0.001) | identity link preferred (ΔAIC = −131) | age×BMI interaction significant but small (p = 0.009, ΔAIC = −4.6) | R² = 0.588
